In [0]:
# ============================================================
# PARAMETERS AND PREVIOUS SILVER BASELINE
# ============================================================

from pyspark.sql import functions as F

dbutils.widgets.text("batch_id", "Batch_01")

batch_id = dbutils.widgets.get("batch_id").strip()

if not batch_id:
    raise ValueError("batch_id cannot be empty")

previous_order_ids = dbutils.jobs.taskValues.get(
    taskKey="capture_silver_baseline",
    key="previous_order_ids"
)

previous_silver_count = dbutils.jobs.taskValues.get(
    taskKey="capture_silver_baseline",
    key="previous_silver_count"
)

previous_order_ids = set(previous_order_ids)

print(f"Validating {batch_id}")

In [0]:
# ============================================================
# LOAD BATCH DATA
# ============================================================

source_table = "final_assesment.bronze.orders_raw"
rejected_table = "final_assesment.dbt_dev.orders_rejected"
silver_table = "final_assesment.dbt_dev.silver_orders"

source_batch = (
    spark.table(source_table)
    .filter(F.col("batch_id") == batch_id)
)

rejected_batch = (
    spark.table(rejected_table)
    .filter(F.col("batch_id") == batch_id)
)

silver_batch = (
    spark.table(silver_table)
    .filter(F.col("batch_id") == batch_id)
)

source_count = source_batch.count()
rejected_count = rejected_batch.count()
silver_processed_count = silver_batch.count()

actual_silver_count = (
    spark.table(silver_table)
    .select("order_id")
    .distinct()
    .count()
)

In [0]:
# ============================================================
# SOURCE AND SILVER RECONCILIATION
# ============================================================

current_batch_order_ids = {
    row["order_id"]
    for row in (
        silver_batch
        .select("order_id")
        .filter(F.col("order_id").isNotNull())
        .distinct()
        .collect()
    )
}

# New order IDs: not present before this batch
new_order_ids = current_batch_order_ids - previous_order_ids

# Updated order IDs: already existed before this batch
updated_order_ids = current_batch_order_ids & previous_order_ids

new_order_count = len(new_order_ids)
updated_order_count = len(updated_order_ids)

expected_silver_count = (
    previous_silver_count + new_order_count
)

# Reconciliation checks
source_reconciliation = (
    source_count == rejected_count + silver_processed_count
)

silver_reconciliation = (
    actual_silver_count == expected_silver_count
)

insert_update_reconciliation = (
    silver_processed_count == new_order_count + updated_order_count
)

print("Reconciliation calculations completed.")

In [0]:
# ============================================================
# SILVER DATA QUALITY CHECKS
# ============================================================

silver_df = spark.table(silver_table)

# 1. Check duplicate order IDs in final Silver
total_silver_rows = silver_df.count()

distinct_silver_orders = (
    silver_df.select("order_id").distinct().count()
)

unique_order_check = (
    total_silver_rows == distinct_silver_orders
)

# 2. Check mandatory fields
mandatory_columns = [
    "order_id",
    "customer_id",
    "product_id"
]

missing_columns = [
    column for column in mandatory_columns
    if column not in silver_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing mandatory Silver columns: {missing_columns}"
    )

null_condition = None

for column in mandatory_columns:
    condition = F.col(column).isNull()

    null_condition = (
        condition if null_condition is None
        else null_condition | condition
    )

null_record_count = silver_df.filter(null_condition).count()

mandatory_fields_check = null_record_count == 0

# 3. Check positive quantities and non-negative prices
numeric_checks = []

if "quantity" in silver_df.columns:
    invalid_quantity_count = (
        silver_df.filter(
            F.col("quantity").isNull() |
            (F.col("quantity") <= 0)
        ).count()
    )
    numeric_checks.append(invalid_quantity_count == 0)

if "unit_price" in silver_df.columns:
    invalid_price_count = (
        silver_df.filter(
            F.col("unit_price").isNull() |
            (F.col("unit_price") < 0)
        ).count()
    )
    numeric_checks.append(invalid_price_count == 0)

numeric_check = all(numeric_checks)

# 4. Final batch checks
quality_checks = [
    ("Source reconciliation", source_reconciliation),
    ("Silver count reconciliation", silver_reconciliation),
    ("Insert/update reconciliation", insert_update_reconciliation),
    ("Unique order IDs", unique_order_check),
    ("Mandatory fields", mandatory_fields_check),
    ("Valid quantity and price", numeric_check)
]

In [0]:
# ============================================================
# FINAL VALIDATION REPORT
# ============================================================

reconciliation_rows = [
    ("Source records", source_count),
    ("Rejected records", rejected_count),
    ("Silver records processed", silver_processed_count),
    ("Previous Silver unique orders", previous_silver_count),
    ("New order IDs", new_order_count),
    ("Existing order IDs updated", updated_order_count),
    ("Expected final Silver orders", expected_silver_count),
    ("Actual final Silver orders", actual_silver_count)
]

reconciliation_df = spark.createDataFrame(
    reconciliation_rows,
    ["Metric", "Count"]
)

quality_df = spark.createDataFrame(
    [
        (name, "PASS" if passed else "FAIL")
        for name, passed in quality_checks
    ],
    ["Validation Check", "Status"]
)

# Display only the two concise reports
display(reconciliation_df)

display(quality_df)

# Overall result
all_passed = all(passed for _, passed in quality_checks)

if all_passed:
    print(f"FINAL RESULT: PASS - {batch_id}")
else:
    print(f"FINAL RESULT: FAIL - {batch_id}")
    raise Exception(
        f"Validation failed for {batch_id}. "
        "Production build must not proceed."
    )